# xLSTM, sLSTM, mLSTMs - with Checkpoints

## Preparation

### Import modules

In [1]:
# Cell 1: Mount & Navigate
from google.colab import drive

drive.mount('/content/drive')

# Go to correct folder
%cd /content/drive/MyDrive/Colab\ Notebooks/thesis/LSTM_Train

# Verify structure
!ls -la ../
# Should show: dataset/  thesis_utils/  LSTM_Train/

!pip install loguru torchxlstm fastparquet

import sys
from pathlib import Path

# Add thesis_utils to path (parent dir)
sys.path.insert(0, '/content/drive/MyDrive/Colab\ Notebooks/thesis')

# Or simpler:
sys.path.insert(0, str(Path.cwd().parent))

<>:20: SyntaxWarning: invalid escape sequence '\ '
<>:20: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-551960470.py:20: SyntaxWarning: invalid escape sequence '\ '
  sys.path.insert(0, '/content/drive/MyDrive/Colab\ Notebooks/thesis')


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/thesis/LSTM_Train
total 29
drwx------ 2 root root 4096 Dec 28 11:56  checkpoints
drwx------ 2 root root 4096 Dec 25 14:57  dataset
drwx------ 2 root root 4096 Dec 25 18:09  GRU_Train
drwx------ 2 root root 4096 Dec 25 14:56  LSTM_Train
drwx------ 2 root root 4096 Dec 26 10:59 'MLSTM   SLSTM'
-rw------- 1 root root  662 Dec 25 15:12  requirements_colab.txt
drwx------ 2 root root 4096 Dec 25 14:56  thesis_utils
drwx------ 2 root root 4096 Dec 25 22:13  XLSTM_Train
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.6 MB/s eta 0:00:00


In [2]:
# Prediction using LSTM, GRU-LSTM, xLSTM

import math
import os
from typing import List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import DataLoader, Dataset, Subset

from sklearn.model_selection import KFold, GroupShuffleSplit

import thesis_utils as tu


In [3]:
CHKPT_DIR = "/content/drive/MyDrive/Colab\ Notebooks/thesis/checkpoints"
os.makedirs(CHKPT_DIR, exist_ok=True)

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipython-input-3523984999.py:1: SyntaxWarning: invalid escape sequence '\ '
  CHKPT_DIR = "/content/drive/MyDrive/Colab\ Notebooks/thesis/checkpoints"


In [4]:
def ckpt_path(serial, fold):
  return os.path.join(CHKPT_DIR, f"{serial}_fold{fold}.pt")

### Configuration

In [5]:
import os

# Model parameters
HORIZON = 1
BATCH_SIZE = 512  # Increased to saturate GPU
EMBEDDING_SIZE = 128
NUM_EPOCHS = 60
HIDDEN_SIZE = 256
N_LAYERS = 3
DROPOUT = 0.05
XLSTM_TYPE = "M"
N_LAGS = 5

# Train parameters
TARGET = "EXPORT_centered"
FEATURES = [
  "contig", "comlang_off", "colony", "smctry",
]
N_SPLITS = 8
PATIENCE = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0
RANDOM_SEED = 16
SUBSAMPLE_ENABLED = True
N_DYADS = 10
XLSTM_LAYERS = "sm"

SANCTION_COLS = ["arms", "military", "trade", "travel", "other"]

# Torch config
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Colab / L4 performance knobs
# ----------------------------
if device.type == "cuda":
  print("GPU:", torch.cuda.get_device_name(0))
  print("CUDA:", torch.version.cuda, "| PyTorch:", torch.__version__)
  !nvidia-smi -L

  # cuDNN autotuner (best when shapes are stable, typical in training)
  torch.backends.cudnn.benchmark = True

  # Better GEMM kernels on Ampere+ (L4 is Ada)
  torch.set_float32_matmul_precision("high")

  # Reduce allocator fragmentation for long runs
  os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Mixed precision (L4 supports bf16 well)
USE_AMP = True
AMP_DTYPE = torch.bfloat16 if (
    device.type == "cuda" and getattr(torch.cuda, "is_bf16_supported", lambda: False)()) else torch.float16

# torch.compile can improve throughput (PyTorch 2.0+)
USE_TORCH_COMPILE = True
COMPILE_MODE = "max-autotune"  # alternatives: "reduce-overhead", "default"

# GradScaler is needed for fp16; for bf16 it should be disabled
USE_GRAD_SCALER = (device.type == "cuda" and USE_AMP and AMP_DTYPE == torch.float16)

print("Using device:", device)
if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))
  # Enable TF32 for faster computing on Ampere+ GPUs
  torch.backends.cuda.matmul.allow_tf32 = True
  torch.backends.cudnn.allow_tf32 = True

dyads_case_study = [
  # "USA_CHN", "CHN_USA",
  # "USA_CAN", "CAN_USA",
  # "DEU_CHN", "CHN_DEU",
  # "USA_DEU", "DEU_USA",
  # "USA_MEX", "MEX_USA",
  # "AUS_CHN", "CHN_AUS",
  # "USA_JPN", "JPN_USA",
  # "DEU_JPN", "JPN_DEU",
  # "USA_AUS", "AUS_USA",
  # "DEU_RUS", "RUS_DEU",
]

Using device: cpu


In [6]:
# Save config
SAVE_ENABLED = False
layers_string = f"({XLSTM_LAYERS})"
SERIAL_NUMBER = (
  f"{XLSTM_TYPE}LSTM"
  f"{layers_string if XLSTM_TYPE == 'X' else ''}"
  f"-{LEARNING_RATE}lr-{DROPOUT}d-{HIDDEN_SIZE}hs-{WEIGHT_DECAY}wd-{BATCH_SIZE}bs-{N_LAYERS}layers-{EMBEDDING_SIZE}es-kfolds{N_SPLITS}-hp"
)
SERIAL_NUMBER = SERIAL_NUMBER.replace(".", "_")
PATH_TO_FOLDER = ""

### Load Data

In [7]:
processed = pd.read_parquet(path="../dataset/processed.parquet", engine="fastparquet")
df: DataFrame = processed.copy(deep=True)

### Sort, shift and compute data

In [8]:
# Sort data by Report + Partner + Year
df["dyad_id"] = df["ISO3_reporter"] + "_" + df["ISO3_partner"]
df = df.sort_values(by=["dyad_id", "Year"], ignore_index=True)

In [9]:
# Remove case study dyad_pairs
mask_keep = ~np.isin(df["dyad_id"], dyads_case_study)
df = df.loc[mask_keep].reset_index(drop=True)

In [10]:
# Sanity check case study pairs
has_overlap = df["dyad_id"].isin(dyads_case_study).any()

if has_overlap:
  print("⚠️ Some case study dyads are present in the DataFrame.")
else:
  print("✅ No case study dyads found in the DataFrame.")

✅ No case study dyads found in the DataFrame.


In [11]:
if SUBSAMPLE_ENABLED:
  dyad_subsample = pd.Series(df["dyad_id"].unique()).sample(n=N_DYADS, random_state=RANDOM_SEED, replace=False)
  df = df[df["dyad_id"].isin(dyad_subsample)]

print(f"Unique dyads: {df["dyad_id"].nunique()}")

Unique dyads: 10


In [12]:
df["sanction"] = (df[SANCTION_COLS]
                  .sum(axis=1)).astype(int)

### Coerce numerical values and convert dyad_id to categorical

In [13]:
num_cols = ["distw", "GDP_reporter", "GDP_partner", "sanction", "contig",
            "comlang_off", "colony", "smctry", "Year", ]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").astype(float)
df = df.dropna(subset=num_cols)

In [14]:
df["Year"] = df["Year"].astype(int)
for col in ["dyad_id"]:
  df[col] = pd.Categorical(df[col], categories=sorted(df[col].unique()))

In [15]:
# Save EXPORT std and median to undo centering
EXPORT_STD = df["EXPORT"].std()
EXPORT_MEDIAN = df["EXPORT"].median()

### Center data

In [16]:
center_columns = ["distw", "GDP_reporter", "GDP_partner", "EXPORT"]
for col in center_columns:
  median = df[col].median()
  std_df = df[col].std()
  df[col + "_centered"] = (df[col] - median) / std_df
FEATURES += ["distw_centered"]

In [17]:
lag_cols = ["GDP_reporter_centered", "GDP_partner_centered", "sanction"]
for col in lag_cols:
  for index in range(1, N_LAGS + 1):
    df[f"{col}_lag{index}"] = df.groupby("dyad_id", observed=True)[col].shift(index)

In [18]:
df = df.dropna()

In [19]:
FEATURES += [f"{c}_lag{index}" for c in lag_cols for index in range(1, N_LAGS + 1)]

## Split data

In [20]:
# Embeddings
dyad_to_idx = { dyad: i for i, dyad in enumerate(df["dyad_id"].cat.categories) }
df["dyad_idx"] = df["dyad_id"].map(dyad_to_idx).astype(int)

In [21]:
# Split into Train, Validation and Test sets
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)

train_idx, test_idx = next(gss.split(df, groups=df["dyad_id"]))
test_df = df.iloc[test_idx]
train_df = df.iloc[train_idx]

train_idx, val_idx = next(gss.split(train_df, groups=train_df["dyad_id"]))
val_df = train_df.iloc[val_idx]
train_df = train_df.iloc[train_idx]

In [22]:
train_df.loc[:, FEATURES] = train_df.loc[:, FEATURES].astype(
  "float32",
  copy=False
)

# Train

## Define Fold and Epoch steps
_For reusability_

In [23]:
# Create KFold object
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

In [24]:



def epoch_step(
    model: nn.Module,
    optimizer: Optimizer,
    criterion: nn.Module,
    scheduler: LRScheduler,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: any,
    scaler: torch.amp.GradScaler,
) -> float:
  # =========================
  # TRAIN
  # =========================
  model.train()

  for X, y, di in train_loader:
    X, y, di = map(lambda t: t.to(device, non_blocking=True), (X, y, di))

    optimizer.zero_grad(set_to_none=True)

    # --- AMP forward ---
    with autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=(getattr(device, "type", "cpu") == "cuda" and USE_AMP)):
      y_pred = model(X, di)

      if not torch.isfinite(y_pred).all():
        print("⚠️ NaN or Inf detected in y_pred — stopping here!")
        return float("inf")

      loss = criterion(y_pred, y)

    if not torch.isfinite(loss):
      print("⚠️ loss is NaN or Inf!")
      return float("inf")

    # --- AMP backward ---
    scaler.scale(loss).backward()

    # IMPORTANT: unscale before clipping
    scaler.unscale_(optimizer)
    clip_grad_norm_(model.parameters(), max_norm=1.0)

    scaler.step(optimizer)
    scaler.update()

    # OneCycleLR MUST step per batch
    scheduler.step()

  # =========================
  # VALIDATION
  # =========================
  model.eval()
  val_losses = []

  with torch.inference_mode():
    for X, y, di in val_loader:
      X, y, di = map(lambda t: t.to(device, non_blocking=True), (X, y, di))
      with autocast(device_type="cuda", dtype=AMP_DTYPE,
                    enabled=(getattr(device, "type", "cpu") == "cuda" and USE_AMP)):
        preds = model(X, di)
        val_losses.append(criterion(preds, y).item())

  val_rmse = math.sqrt(sum(val_losses) / len(val_losses))
  return val_rmse

In [25]:
from torch.amp import GradScaler, autocast
import copy
import inspect


def _make_loader(subset, *, batch_size: int, shuffle: bool, n_workers: int) -> DataLoader:
  """Colab-friendly DataLoader with aggressive prefetch + pinned memory."""
  kwargs = dict(
    batch_size=batch_size,
    shuffle=shuffle,
    num_workers=n_workers,
    pin_memory=True,
    persistent_workers=(n_workers > 0),
  )

  # Keep shapes static for better GPU kernels / torch.compile (when possible)
  if shuffle:
    kwargs["drop_last"] = True

  # Only valid when num_workers > 0
  if n_workers > 0:
    kwargs["prefetch_factor"] = 4

  # Available in newer PyTorch; safe-guard for older runtimes
  if "pin_memory_device" in inspect.signature(DataLoader).parameters:
    kwargs["pin_memory_device"] = "cuda"

  return DataLoader(subset, **kwargs)


# Define fold step
def fold_step(
    fold: int,
    train_idx: List,
    val_idx: List,
    dataset: Dataset,
    batch_size: int,
    num_epochs: int,
    model: nn.Module,
    device: any,
    optimizer: Optimizer,
    criterion: nn.Module,
    serial: str,
) -> (float, dict):
  # Colab CPU resources vary; too many workers can hurt more than help.
  # L4 runtimes typically benefit from 4-8 workers.
  n_workers = min(8, (os.cpu_count() or 2))

  train_loader = _make_loader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True, n_workers=n_workers)
  val_loader = _make_loader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False, n_workers=n_workers)

  scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=7e-4,
    epochs=num_epochs,
    steps_per_epoch=len(train_loader),
    div_factor=25,
    final_div_factor=100,
    pct_start=0.3,
    anneal_strategy="cos",
    three_phase=False,
  )

  ckpt_file = ckpt_path(serial, fold)
  start_epoch = 0
  best_rmse = float("inf")
  best_state = copy.deepcopy(model.state_dict())
  patience_left = 10

  scaler = GradScaler("cuda", enabled=USE_GRAD_SCALER)

  if os.path.exists(ckpt_file):
    print(f"🔄 Resuming from checkpoint: {ckpt_file}")
    ckpt = torch.load(ckpt_file, map_location=device)

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])

    # Re-create scaler and load its state (state_dict, not the scaler itself)
    if "scaler_state" in ckpt and ckpt["scaler_state"] is not None:
      try:
        scaler.load_state_dict(ckpt["scaler_state"])
      except Exception as e:
        print(f"⚠️ Could not restore GradScaler state: {e}")

    start_epoch = int(ckpt.get("epoch", -1)) + 1
    best_rmse = float(ckpt.get("best_rmse", best_rmse))
    best_state = ckpt.get("best_state", best_state)

  print(f"Start epoch train for fold {fold} (from epoch {start_epoch})")

  for epoch in range(start_epoch, num_epochs):
    val_rmse = epoch_step(
      model=model,
      optimizer=optimizer,
      criterion=criterion,
      scheduler=scheduler,
      train_loader=train_loader,
      val_loader=val_loader,
      device=device,
      scaler=scaler,
    )

    print(f"Epoch {epoch + 1:02d}/{num_epochs}  |  val RMSE: {val_rmse:.4f}")

    improved = val_rmse < best_rmse - 1e-4
    if improved:
      best_rmse = val_rmse
      best_state = copy.deepcopy(model.state_dict())
      patience_left = 10
    else:
      patience_left -= 1

    # NOTE: Checkpointing every epoch can become a bottleneck. If you want more speed,
    # set CKPT_SAVE_EVERY > 1 and only save periodically (or only on improvements).
    torch.save(
      {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "best_rmse": best_rmse,
        "best_state": best_state,
      },
      ckpt_file,
    )

    if patience_left <= 0:
      print("⏹️ Early stopping triggered.")
      break

  # =========================
  # FINAL EVAL ON VALIDATION
  # =========================
  model.load_state_dict(best_state)
  model.eval()

  preds_centered, truth_centered = [], []

  with torch.inference_mode():
    for X, y, di in val_loader:
      X, di = map(lambda t: t.to(device, non_blocking=True), (X, di))
      with autocast(device_type="cuda", dtype=AMP_DTYPE,
                    enabled=(getattr(device, "type", "cpu") == "cuda" and USE_AMP)):
        out = model(X, di)
      preds_centered.append(out.float().cpu())
      truth_centered.append(y)

  preds_centered = torch.cat(preds_centered).numpy()
  truth_centered = torch.cat(truth_centered).numpy()

  # Convert back to raw scale
  preds_raw = preds_centered * EXPORT_STD + EXPORT_MEDIAN
  truth_raw = truth_centered * EXPORT_STD + EXPORT_MEDIAN

  rmse_centered = tu.rmse(truth_centered, preds_centered)
  mae_centered = tu.mae(truth_centered, preds_centered)
  rmae_centered = tu.rmae(truth_centered, preds_centered)
  pseudo_r2_centered = tu.pseudo_r2(truth_centered, preds_centered)

  print(
    f"Fold {fold} CENTERED  RMSE {rmse_centered:.4f} | MAE {mae_centered:.4f} | R² {pseudo_r2_centered:.4f} | RMAE {rmae_centered:.4f}"
  )

  rmse_raw = tu.rmse(truth_raw, preds_raw)
  mae_raw = tu.mae(truth_raw, preds_raw)
  rmae_raw = tu.rmae(truth_raw, preds_raw)
  pseudo_r2_raw = tu.pseudo_r2(truth_raw, preds_raw)

  print(
    f"Fold {fold} RAW       RMSE {rmse_raw:.4f} | MAE {mae_raw:.4f} | R² {pseudo_r2_raw:.4f} | RMAE {rmae_raw:.4f}"
  )

  return (
    { "RMSE": rmse_centered, "MAE": mae_centered, "R2": pseudo_r2_centered, "RMAE": rmae_centered },
    { "RMSE": rmse_raw, "MAE": mae_raw, "R2": pseudo_r2_raw, "RMAE": rmae_raw },
    copy.deepcopy(best_state),
  )


## Train Raw dataset

### Split dataset

In [26]:
# Convert df_scaled to pytorch Tensor
dataset, dyad_to_idx = tu.make_panel_datasets_dyad(
  data=df,
  features=FEATURES,
  target=TARGET,
  horizon=HORIZON,
)

In [27]:
import os

n_workers = min(8, (os.cpu_count() or 2))

train_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=True,
  drop_last=True,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)
val_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)
test_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)

if n_workers > 0:
  train_kwargs["prefetch_factor"] = 4
  val_kwargs["prefetch_factor"] = 4
  test_kwargs["prefetch_factor"] = 4

train_loader = DataLoader(Subset(dataset, train_idx), **train_kwargs)
val_loader = DataLoader(Subset(dataset, val_idx), **val_kwargs)
test_loader = DataLoader(Subset(dataset, test_idx), **test_kwargs)


### Train model

In [28]:
# Save best train iteration
best_fold_state = None
best_fold_rmse = float("inf")

metrics_per_fold = {
  "RMSE": [],
  "MAE": [],
  "R2": [],
  "RMAE": [],
}

metrics_per_fold_raw = {
  "RMSE": [],
  "MAE": [],
  "R2": [],
  "RMAE": [],
}

In [29]:
for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(dataset))), 1):

  ckpt_file = ckpt_path(SERIAL_NUMBER, fold)
  if os.path.exists(ckpt_file):
    ckpt = torch.load(ckpt_file, map_location="cpu")
    if ckpt["epoch"] >= NUM_EPOCHS - 1:
      print(f"✅ Fold {fold} already completed — skipping.")
      continue

  print(f"=== FOLD {fold}/{N_SPLITS} ===")

  model = tu.DyadXLSTM(
    n_features=len(FEATURES),
    n_dyads=len(dyad_to_idx),
    embed_dim=EMBEDDING_SIZE,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    horizon=HORIZON,
    type=XLSTM_TYPE,
    layers=XLSTM_LAYERS,
    n_layers=N_LAYERS,
  ).to(device=device)

  # torch.compile can improve throughput (PyTorch 2.0+)
  if USE_TORCH_COMPILE and hasattr(torch, "compile") and device.type == "cuda":
    try:
      model = torch.compile(model, mode=COMPILE_MODE)
      print(f"Model compiled with torch.compile(mode={COMPILE_MODE!r})")
    except Exception as e:
      print(f"Could not compile model: {e}")

  criterion = nn.SmoothL1Loss(beta=0.5)
  adamw_kwargs = dict(lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98), eps=1e-8)
  if device.type == "cuda":
    try:
      optimizer = optim.AdamW(model.parameters(), **adamw_kwargs, fused=True)
      print("Using fused AdamW")
    except TypeError:
      optimizer = optim.AdamW(model.parameters(), **adamw_kwargs)
  else:
    optimizer = optim.AdamW(model.parameters(), **adamw_kwargs)

  fold_metrics, fold_metrics_raw, best_state = fold_step(fold=fold,
                                                         train_idx=train_idx,
                                                         val_idx=val_idx,
                                                         dataset=dataset,
                                                         batch_size=BATCH_SIZE,
                                                         num_epochs=NUM_EPOCHS,
                                                         model=model,
                                                         device=device,
                                                         optimizer=optimizer,
                                                         criterion=criterion,
                                                         serial=SERIAL_NUMBER)
  # scheduler=scheduler)
  if fold_metrics["RMSE"] < best_fold_rmse:
    best_fold_rmse = fold_metrics["RMSE"]
    best_fold_state = copy.deepcopy(best_state)

  for k, v in fold_metrics.items():
    metrics_per_fold[k].append(v)

  for k, v in fold_metrics_raw.items():
    metrics_per_fold_raw[k].append(v)


=== FOLD 1/8 ===
Model compiled with torch.compile()
Start epoch train for fold 1


/tmp/ipython-input-781372058.py:55: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler("cuda")
/tmp/ipython-input-781372058.py:58: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler("cuda")
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 01/60  |  val RMSE: 0.6018
Epoch 02/60  |  val RMSE: 0.5651


/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:270: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Epoch 03/60  |  val RMSE: 0.5165
Epoch 04/60  |  val RMSE: 0.4434
Epoch 05/60  |  val RMSE: 0.3767
Epoch 06/60  |  val RMSE: 0.3368
Epoch 07/60  |  val RMSE: 0.3871
Epoch 08/60  |  val RMSE: 0.5173
Epoch 09/60  |  val RMSE: 0.6645
Epoch 10/60  |  val RMSE: 0.7635
Epoch 11/60  |  val RMSE: 0.7508
Epoch 12/60  |  val RMSE: 0.6800
Epoch 13/60  |  val RMSE: 0.5987
Epoch 14/60  |  val RMSE: 0.5500
Epoch 15/60  |  val RMSE: 0.5809
Epoch 16/60  |  val RMSE: 0.6383
Epoch 17/60  |  val RMSE: 0.6324
Epoch 18/60  |  val RMSE: 0.5556
Epoch 19/60  |  val RMSE: 0.4959
Epoch 20/60  |  val RMSE: 0.5574
Epoch 21/60  |  val RMSE: 0.5745
Epoch 22/60  |  val RMSE: 0.5753
Epoch 23/60  |  val RMSE: 0.5430
Epoch 24/60  |  val RMSE: 0.5050
Epoch 25/60  |  val RMSE: 0.4796
Epoch 26/60  |  val RMSE: 0.5526
Epoch 27/60  |  val RMSE: 0.5970
Epoch 28/60  |  val RMSE: 0.6014
Epoch 29/60  |  val RMSE: 0.5834
Epoch 30/60  |  val RMSE: 0.5157
Epoch 31/60  |  val RMSE: 0.5383
Epoch 32/60  |  val RMSE: 0.5298
Epoch 33/6

/tmp/ipython-input-781372058.py:55: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler("cuda")
/tmp/ipython-input-781372058.py:58: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = GradScaler("cuda")
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch 01/60  |  val RMSE: 0.5268
Epoch 02/60  |  val RMSE: 0.4852
Epoch 03/60  |  val RMSE: 0.4297
Epoch 04/60  |  val RMSE: 0.3640
Epoch 05/60  |  val RMSE: 0.3419
Epoch 06/60  |  val RMSE: 0.4372
Epoch 07/60  |  val RMSE: 0.6297
Epoch 08/60  |  val RMSE: 0.8055
Epoch 09/60  |  val RMSE: 0.9175
Epoch 10/60  |  val RMSE: 0.9190
Epoch 11/60  |  val RMSE: 0.8445
Epoch 12/60  |  val RMSE: 0.7273
Epoch 13/60  |  val RMSE: 0.6321
Epoch 14/60  |  val RMSE: 0.6279
Epoch 15/60  |  val RMSE: 0.6807
Epoch 16/60  |  val RMSE: 0.7126
Epoch 17/60  |  val RMSE: 0.7167
Epoch 18/60  |  val RMSE: 0.6837
Epoch 19/60  |  val RMSE: 0.6440
Epoch 20/60  |  val RMSE: 0.5921
Epoch 21/60  |  val RMSE: 0.5681
Epoch 22/60  |  val RMSE: 0.5626
Epoch 23/60  |  val RMSE: 0.5922
Epoch 24/60  |  val RMSE: 0.6780
Epoch 25/60  |  val RMSE: 0.6328
Epoch 26/60  |  val RMSE: 0.4866
Epoch 27/60  |  val RMSE: 0.6867
Epoch 28/60  |  val RMSE: 0.5286
Epoch 29/60  |  val RMSE: 0.4206
Epoch 30/60  |  val RMSE: 0.5893
Epoch 31/6

KeyboardInterrupt: 

## Save Model

In [ ]:
torch.save({
  "model_state_dict": best_fold_state,
  "model_hyperparams": {
    "n_features": len(FEATURES),
    "n_dyads": len(dyad_to_idx),
    "n_layers": N_LAYERS,
    "embed_dim": EMBEDDING_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "dropout": DROPOUT,
    "horizon": HORIZON,
    "type": XLSTM_TYPE,
    "layers": XLSTM_LAYERS,
  },
  "dyad_to_idx": dyad_to_idx,
  "feature_names": FEATURES,
}, PATH_TO_FOLDER + SERIAL_NUMBER + ".pt")
print("Saved model to", PATH_TO_FOLDER + SERIAL_NUMBER + ".pt")

In [ ]:
def summarize(xs):
  xs = np.asarray(xs, dtype=float)
  n = xs.size
  mean = xs.mean()
  std = xs.std(ddof=1)  # sample std
  se = std / math.sqrt(n)
  try:
    from scipy.stats import t
    tcrit = t.ppf(0.975, df=n - 1)
  except Exception:
    tcrit = 1.96  # normal approx≈
  ci95 = tcrit * se
  return mean, std, ci95

In [ ]:
print("\n=== Cross-fold CENTERED summary ===")
for name in ["MAE", "RMSE", "R2", "RMAE"]:
  mean, std, ci = summarize(metrics_per_fold[name])
  print(f"{name:>5}: {mean:.4f} ± {std:.4f}  (95% CI ±{ci:.4f})")

print("\n=== Cross-fold RAW summary ===")
for name in ["MAE", "RMSE", "R2", "RMAE"]:
  mean, std, ci = summarize(metrics_per_fold_raw[name])
  print(f"{name:>5}: {mean:.4f} ± {std:.4f}  (95% CI ±{ci:.4f})")